In [18]:
import pandas as pd
import numpy as np
import pickle
import geopandas as gpd
from itertools import product
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import os
import torch

In [3]:
path_hex = "/mnt/raid1/MAAT/08.accessibility/Copenhagen/"
with open(path_hex + "zones_Copenhagen.pkl", 'rb') as f:
    hexes = pickle.load(f)

tazes = hexes[['taz_zoneid', 'geometry']].dissolve(by='taz_zoneid')
tazes = tazes.to_crs(epsg=25832)

# Ordered Taz IDS
TAZ_IDS = sorted(hexes['taz_zoneid'].unique())
taz_to_idx = {taz_id: i for i, taz_id in enumerate(TAZ_IDS)}

In [4]:
def create_edge_index(tazes, taz_to_idx):
    tazes_touching = gpd.sjoin(tazes, tazes, predicate='touches') #only keeps left geometry
    tazes_touching = tazes_touching.reset_index()

    tazes_touching['geometry_right']   = tazes_touching['taz_zoneid_right'].map(tazes['geometry'])
    tazes_touching['shared_boundary']  = tazes_touching['geometry'].intersection(tazes_touching['geometry_right'])
    tazes_touching['boundary_weight']  = tazes_touching['shared_boundary'].length
    
    src = tazes_touching['taz_zoneid_left'].map(taz_to_idx)
    dst = tazes_touching['taz_zoneid_right'].map(taz_to_idx)

    edge_index_np = np.array([src.to_numpy(), dst.to_numpy()])  # combine into one ndarray first - torch.tensor() on a list of ndarrays is slow
    edge_index = torch.tensor(edge_index_np, dtype=torch.long)
    edge_weight = torch.tensor(tazes_touching['boundary_weight'].to_numpy(), dtype=torch.float32)

    return edge_index, edge_weight


def create_polygon_metrics(tazes):
    area      = torch.tensor(tazes.geometry.area.reindex(TAZ_IDS).to_numpy(), dtype=torch.float32)
    perimeter = torch.tensor(tazes.geometry.length.reindex(TAZ_IDS).to_numpy(), dtype=torch.float32)

    return area, perimeter


In [5]:
ei, ew = create_edge_index(tazes, taz_to_idx)
a, p = create_polygon_metrics(tazes)


In [6]:

TRANSPORT_MODES = ['CAR', 'BICYCLE', 'ON_FOOT']
POI_CATEGORIES  = ['cultural', 'education', 'green_space', 'health',
                   'public_spaces', 'public_transportation', 'sports']


path_access = "/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet_taz"
ds = load_dataset("parquet", data_files=os.path.join(path_access, "*.parquet"))
ds = ds['train']

ds.set_format(type='torch')   # no columns= restriction — ALL columns become tensors            # dict with every column, each as a tensor
ds = ds[:]


dry_baseline_path = "/mnt/raid1/MAAT/20.surrogate_data/cph/accessibility_withassignment/parquet_taz_baseline"
dry_baseline = load_dataset("parquet", data_files=os.path.join(dry_baseline_path, "*.parquet"))
dry_baseline = dry_baseline['train']

dry_baseline.set_format(type='torch')   # no columns= restriction — ALL columns become tensors            # dict with every column, each as a tensor
dry_baseline = dry_baseline[:]


x_cols = [f"water_depths_{mode}" for mode in TRANSPORT_MODES]
y_cols = [f"cumulative_accessibility_{mode}_{poi}"
          for mode, poi in product(TRANSPORT_MODES, POI_CATEGORIES)]

x_dynamic = torch.stack([ds[c] for c in x_cols], dim=-1)   # [1010, 277, 3]

x_static = torch.stack([a,p], dim=-1)
x_static = x_static.unsqueeze(dim=0)
x_static = x_static.expand(x_dynamic.shape[0],-1,-1)

x = torch.cat((x_dynamic, x_static), dim=-1)


y_absolute = torch.stack([ds[c] for c in y_cols], dim=-1)   # [1010, 277, 21]

y_dry = torch.stack([dry_baseline[c] for c in y_cols], dim=-1)

# DECISION: currently predicting the deviation from the dry (no-flood) baseline,
# not the absolute accessibility value. To switch back to predicting the absolute
# value later, just use y = y_absolute instead of the line below.
y = y_absolute - y_dry



In [7]:
y.shape

torch.Size([1010, 277, 21])

In [8]:
x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, test_size=0.2, random_state=13)
x_train, x_val, y_train, y_val = train_test_split(x_tmp, y_tmp, test_size=0.125, random_state=13)


x_mean = x_train.mean(dim=(0,1))
x_std  = x_train.std(dim=(0,1)).clamp(1e-6)

y_mean = y_train.mean(dim=(0,1))
y_std = y_train.std(dim=(0,1)).clamp(1e-6)

x_train_n = (x_train - x_mean) / x_std
x_val_n   = (x_val   - x_mean) / x_std
x_test_n  = (x_test  - x_mean) / x_std

y_train_n = (y_train - y_mean) / y_std
y_val_n   = (y_val   - y_mean) / y_std
y_test_n  = (y_test  - y_mean) / y_std




In [19]:
train_data = [
    Data(x=x_train_n[i], y=y_train_n[i], edge_index=ei, edge_weight=ew) for i in range(x_train_n.shape[0])
]
val_data = [
    Data(x=x_val_n[i], y=y_val_n[i], edge_index=ei, edge_weight=ew) for i in range(x_val_n.shape[0])
]   

test_data = [
    Data(x=x_test_n[i], y=y_test_n[i], edge_index=ei, edge_weight=ew) for i in range(x_test_n.shape[0])
] 

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


In [16]:
len(train_data[0])

4

In [17]:
train_data[0]

Data(x=[277, 5], edge_index=[2, 1456], y=[277, 21], edge_weight=[1456])